In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

In [3]:
# Saving the URL for the API request and the header to retrieve csv data
url = 'https://esploradati.istat.it/SDMXWS/rest/data/41_983'
header = {'Accept': 'application/vnd.sdmx.data+csv;version=1.0.0'}

# make the HTTP request
r = requests.get(url, headers=header)

# Check the status code
r.status_code


ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [7]:
# converting the file into a text
df_text = r.text
print(df_text[:100])

DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_


In [3]:
'''# Saving the file into a csv format
with open('df_csv', 'w', encoding='utf-8') as f:
    f.write(df_text)'''

# Coverting the csv file into a pandas dataframe
df = pd.read_csv('C:/Users/andre/OneDrive/Desktop/Boolean/Final Project/df_csv')
df.head()

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,1001,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,1001,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,1001,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,1001,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,1001,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Checking for NaN values
df.isna().sum()

DATAFLOW                 0
FREQ                     0
REF_AREA                 0
DATA_TYPE                0
RESULT                   0
TIME_PERIOD              0
OBS_VALUE                0
OBS_STATUS          573552
NOTE_DS             573552
NOTE_REF_AREA       573552
NOTE_DATA_TYPE      573552
NOTE_RESULT         573552
NOTE_TIME_PERIOD    573552
BASE_PER            573552
UNIT_MEAS           573552
UNIT_MULT           573552
dtype: int64

In [6]:
for col, row in df.items():
    print(col, row.unique())

DATAFLOW <StringArray>
['IT1:41_983(1.0)']
Length: 1, dtype: str
FREQ <StringArray>
['A']
Length: 1, dtype: str
REF_AREA [  1001   1002   1003 ... 111105 111106 111107]
DATA_TYPE <StringArray>
['KILLINJ', 'ROADACC']
Length: 2, dtype: str
RESULT <StringArray>
['F', 'M', '9']
Length: 3, dtype: str
TIME_PERIOD [2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013 2014
 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]
OBS_VALUE [  10    7   13 ... 1081  880  969]
OBS_STATUS [nan]
NOTE_DS [nan]
NOTE_REF_AREA [nan]
NOTE_DATA_TYPE [nan]
NOTE_RESULT [nan]
NOTE_TIME_PERIOD [nan]
BASE_PER [nan]
UNIT_MEAS [nan]
UNIT_MULT [nan]


In [4]:
# Creating a new dataframe by removing NaN columns and columns with the same data on it
df_filtered = df[['REF_AREA', 'DATA_TYPE', 'RESULT', 'TIME_PERIOD', 'OBS_VALUE']]

In [5]:
df_filtered.head()

,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE
0,1001,KILLINJ,F,2001,10
1,1001,KILLINJ,F,2002,10
2,1001,KILLINJ,F,2003,7
3,1001,KILLINJ,F,2004,13
4,1001,KILLINJ,F,2005,2


In [31]:
# Web scraping the data to get info about area population and cities
chrome_options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options = chrome_options)
driver.get('https://situas.istat.it/web/#/territorio/body?id=74&dateFrom=2020-12-31')

In [32]:
# Finding column names
columns = driver.find_elements(By.CLASS_NAME, 'mr-1')
column_names = [name.text for name in columns]
column_names

['Codice Ripartizione geografica',
 'Codice Regione',
 'Codice Provincia (Storico)',
 'Codice Provincia/Uts',
 'Codice Comune (alfanumerico)',
 'Codice Comune (numerico)',
 'Comune',
 'Comune (dizione straniera)',
 'Sigla automobilistica',
 'Capoluogo di Provincia/Uts',
 'Capoluogo di Regione',
 'Popolazione legale',
 'Anno Censimento',
 'Superficie (Kmq)',
 'Anno (Superficie)',
 'Popolazione residente',
 'Anno (Popolazione residente)']

In [33]:
# Creating the dataframe
df_comuni = pd.DataFrame(columns=column_names)
df_comuni

,Codice Ripartizione geografica,Codice Regione,Codice Provincia (Storico),Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente)


In [35]:
# load 100 comuni per page
driver.find_element(By.XPATH, '//*[@id="tableid"]/div/div[2]/div/div/nav/ul/li[11]/div').click()


In [40]:
driver.find_element(By.XPATH, '//*[@id="tableid"]/div/div[2]/div/div/nav/ul/li[11]/div/div[1]/span[2]').click()

In [41]:
# Finding rows
rows = driver.find_elements(By.CSS_SELECTOR, 'tbody.ant-table-tbody tr.ant-table-row')
data =[]
for row in rows:
    cells = row.find_elements(By.CSS_SELECTOR, 'td.ant-table-cell')
    row_data = [cell.text.strip() for cell in cells]
    data.append(row_data)

df_comuni = pd.DataFrame(data, columns=column_names)


In [42]:
df_comuni

,Codice Ripartizione geografica,Codice Regione,Codice Provincia (Storico),Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente)
0,1,01,001,201,001001,1001,Agliè,,TO,0,0,2644,2011,"13,1462",2020,2545,2020
1,1,01,001,201,001002,1002,Airasca,,TO,0,0,3819,2011,"15,7393",2020,3633,2020
2,1,01,001,201,001003,1003,Ala di Stura,,TO,0,0,462,2011,"46,3315",2020,459,2020
3,1,01,001,201,001004,1004,Albiano d'Ivrea,,TO,0,0,1791,2011,"11,7314",2020,1638,2020
4,1,01,001,201,001006,1006,Almese,,TO,0,0,6303,2011,"17,8756",2020,6355,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1,01,001,201,001097,1097,Cumiana,,TO,0,0,7825,2011,"60,7335",2020,7787,2020
96,1,01,001,201,001098,1098,Cuorgnè,,TO,0,0,10084,2011,"19,3134",2020,9413,2020
97,1,01,001,201,001099,1099,Druento,,TO,0,0,8436,2011,"27,5395",2020,8822,2020
98,1,01,001,201,001100,1100,Exilles,,TO,0,0,266,2011,"46,5474",2020,244,2020


In [43]:
# For cicle to estrapolate data within each page
def scrape_page(page):
    driver.find_elements(By.LINK_TEXT, str(page))[0].click()
    time.sleep(3) # Wait for 3 seconds

    rows = driver.find_elements(By.CSS_SELECTOR, 'tbody.ant-table-tbody tr.ant-table-row')
    data = []

    for row in rows:
        cells = row.find_elements(By.CSS_SELECTOR, 'td.ant-table-cell')
        row_data = [cell.text.strip() for cell in cells]
        data.append(row_data)

    return pd.DataFrame(data, columns=column_names)

df_list = []
pages = [p for p in range(2, 81)] # changing the range according to the last page saved

for p in pages:
    df_list.append(scrape_page(p))
    if p %20 == 0: # save the df every 20 pages
        df_comuni = pd.concat(df_list, ignore_index=True)
        print(p)

df_comuni =pd.concat(df_list, ignore_index=True)

20
40
60
80


In [48]:
df_comuni.shape

(7803, 17)